E-10: Implementare il pre-addestramento e verificare che funzioni con due controlli:
1) la loss di ricostruzione scende sotto quella di un predittore
banale che restituisce sempre la media;
2) un probe lineare per label_20 sopra z_cls congelato fa meglio dello stesso probe sopra un encoder a pesi casuali.
Se fallisce, il pre-addestramento non ha imparato nulla di utile e non ha senso proseguire: torniamo indietro e
capiamo perché.

In [ ]:
# E-10 — Masking del 30% dei pacchetti validi

def create_pretraining_mask(pad_mask, mask_ratio=0.30):
    """
    pad_mask:
        True  = padding
        False = pacchetto valido

    Restituisce:
        mask = True nelle posizioni da ricostruire
    """

    valid = ~pad_mask

    # Probabilità casuale di selezionare ogni posizione
    random_values = torch.rand(
        pad_mask.shape,
        device=pad_mask.device
    )

    # Selezioniamo circa il 30% delle posizioni valide
    mask = (random_values < mask_ratio) & valid

    # Assicuriamoci che ogni flusso abbia almeno
    # una posizione mascherata
    for i in range(mask.size(0)):

        if not mask[i].any() and valid[i].any():

            valid_positions = torch.where(valid[i])[0]

            random_position = valid_positions[
                torch.randint(
                    len(valid_positions),
                    (1,)
                )
            ]

            mask[i, random_position] = True

    return mask


print("Funzione di masking creata correttamente.")

Funzione di masking creata correttamente.


In [ ]:
# E-10 — Test del masking

# Creiamo un batch di esempio
x_test, pad_mask_test, _ = next(iter(extract_train_loader))

mask_test = create_pretraining_mask(
    pad_mask_test,
    mask_ratio=0.30
)

valid_test = ~pad_mask_test

# Percentuale di posizioni valide mascherate
percentuale = (
    mask_test.sum().item()
    / valid_test.sum().item()
    * 100
)

# Controlliamo che nessun padding sia stato mascherato
padding_mascherato = (
    mask_test & pad_mask_test
).sum().item()

# Controlliamo che ogni flusso abbia almeno una posizione mascherata
flussi_senza_mask = (
    mask_test.sum(dim=1) == 0
).sum().item()

print(f"Percentuale posizioni valide mascherate: {percentuale:.2f}%")
print("Posizioni di padding mascherate:", padding_mascherato)
print("Flussi senza posizioni mascherate:", flussi_senza_mask)

if padding_mascherato == 0 and flussi_senza_mask == 0:
    print("✓ Test del masking SUPERATO")
else:
    print("✗ Test del masking FALLITO")

Percentuale posizioni valide mascherate: 29.93%
Posizioni di padding mascherate: 0
Flussi senza posizioni mascherate: 0
✓ Test del masking SUPERATO


In [ ]:
# E-10 — Testa di ricostruzione

class ReconstructionHead(nn.Module):

    def __init__(self, d=64):
        super().__init__()

        self.size_head = nn.Linear(d, 1)
        self.iat_head = nn.Linear(d, 1)
        self.direction_head = nn.Linear(d, 1)

    def forward(self, h):
        """
        h: (B, n_pkt, d)

        Restituisce:
        size       -> (B, n_pkt)
        iat        -> (B, n_pkt)
        direction  -> (B, n_pkt)
        """

        size = self.size_head(h).squeeze(-1)
        iat = self.iat_head(h).squeeze(-1)
        direction = self.direction_head(h).squeeze(-1)

        return size, iat, direction


# Creiamo la testa
reconstruction_head = ReconstructionHead(d=64)

print(reconstruction_head)

ReconstructionHead(
  (size_head): Linear(in_features=64, out_features=1, bias=True)
  (iat_head): Linear(in_features=64, out_features=1, bias=True)
  (direction_head): Linear(in_features=64, out_features=1, bias=True)
)


In [ ]:
# E-10 — Loss di ricostruzione

def reconstruction_loss(
    pred_size,
    pred_iat,
    pred_direction,
    target_x,
    mask
):
    """
    Calcola la loss solo sulle posizioni mascherate.

    pred_size:      (B, 20)
    pred_iat:       (B, 20)
    pred_direction: (B, 20)
    target_x:       (B, 20, 3)
                    [:,:,0] = direction
                    [:,:,1] = log size
                    [:,:,2] = log IAT
    mask:           (B, 20)
                    True = posizione da ricostruire
    """

    # Target
    target_direction = (target_x[:, :, 0] > 0).float()
    target_size = target_x[:, :, 1]
    target_iat = target_x[:, :, 2]

    # Selezioniamo solo le posizioni mascherate
    pred_size_masked = pred_size[mask]
    pred_iat_masked = pred_iat[mask]
    pred_direction_masked = pred_direction[mask]

    target_size_masked = target_size[mask]
    target_iat_masked = target_iat[mask]
    target_direction_masked = target_direction[mask]

    # Loss per size e IAT
    loss_size = nn.functional.mse_loss(
        pred_size_masked,
        target_size_masked
    )

    loss_iat = nn.functional.mse_loss(
        pred_iat_masked,
        target_iat_masked
    )

    # Loss per direction
    loss_direction = nn.functional.binary_cross_entropy_with_logits(
        pred_direction_masked,
        target_direction_masked
    )

    # Loss totale
    loss = loss_size + loss_iat + loss_direction

    return loss, loss_size, loss_iat, loss_direction


print("✓ Funzione reconstruction_loss creata correttamente.")

✓ Funzione reconstruction_loss creata correttamente.


In [ ]:
# E-10 — Baseline: predizione sempre uguale alla media del training set

# Accumulatori per calcolare le medie sui soli pacchetti validi
sum_size = 0.0
sum_iat = 0.0
sum_direction = 0.0
count_valid = 0

for x, pad_mask, _ in extract_train_loader:

    valid = ~pad_mask

    sum_size += x[:, :, 1][valid].sum().item()
    sum_iat += x[:, :, 2][valid].sum().item()

    # Direction: -1/+1 -> 0/1
    direction = (x[:, :, 0] > 0).float()
    sum_direction += direction[valid].sum().item()

    count_valid += valid.sum().item()

# Medie del training set
mean_size = sum_size / count_valid
mean_iat = sum_iat / count_valid
mean_direction = sum_direction / count_valid

print(f"Media log size: {mean_size:.6f}")
print(f"Media log IAT:  {mean_iat:.6f}")
print(f"Media direction (prob. +1): {mean_direction:.6f}")
print(f"Numero pacchetti validi: {count_valid}")

print("✓ Medie della baseline calcolate sul training set.")

Media log size: -0.000000
Media log IAT:  -0.000000
Media direction (prob. +1): 0.229891
Numero pacchetti validi: 2588260
✓ Medie della baseline calcolate sul training set.


In [ ]:
# E-10 — Calcolo della loss della baseline

# Prendiamo un batch dal training set
x_batch, pad_mask_batch, _ = next(iter(extract_train_loader))

# Creiamo una maschera del 30% dei pacchetti validi
mask_batch = create_pretraining_mask(
    pad_mask_batch,
    mask_ratio=0.30
)

# Target
target_size = x_batch[:, :, 1]
target_iat = x_batch[:, :, 2]
target_direction = (x_batch[:, :, 0] > 0).float()

# Consideriamo solo le posizioni mascherate
target_size_masked = target_size[mask_batch]
target_iat_masked = target_iat[mask_batch]
target_direction_masked = target_direction[mask_batch]

# Baseline: prediciamo sempre la media del training set
baseline_size = torch.full_like(
    target_size_masked,
    mean_size
)

baseline_iat = torch.full_like(
    target_iat_masked,
    mean_iat
)

baseline_direction = torch.full_like(
    target_direction_masked,
    mean_direction
)

# Loss della baseline
baseline_loss_size = nn.functional.mse_loss(
    baseline_size,
    target_size_masked
)

baseline_loss_iat = nn.functional.mse_loss(
    baseline_iat,
    target_iat_masked
)

baseline_loss_direction = nn.functional.binary_cross_entropy(
    baseline_direction,
    target_direction_masked
)

baseline_loss = (
    baseline_loss_size
    + baseline_loss_iat
    + baseline_loss_direction
)

print(f"Baseline loss totale:    {baseline_loss.item():.6f}")
print(f"Baseline loss size:      {baseline_loss_size.item():.6f}")
print(f"Baseline loss IAT:       {baseline_loss_iat.item():.6f}")
print(f"Baseline loss direction: {baseline_loss_direction.item():.6f}")
print(f"Posizioni mascherate:    {mask_batch.sum().item()}")

if torch.isfinite(baseline_loss):
    print("✓ Test della baseline SUPERATO")
else:
    print("✗ Test della baseline FALLITO")



Baseline loss totale:    1.641549
Baseline loss size:      0.567892
Baseline loss IAT:       0.223082
Baseline loss direction: 0.850575
Posizioni mascherate:    1120
✓ Test della baseline SUPERATO


In [ ]:
# E-10 — Preparazione del pretraining

# Mettiamo il modello e la reconstruction head in modalità training
model.train()
reconstruction_head.train()

# Ottimizzatore
optimizer_pretrain = torch.optim.Adam(
    list(model.parameters()) + list(reconstruction_head.parameters()),
    lr=1e-3
)

print("✓ Modello pronto per il pretraining.")
print(f"Learning rate: {1e-3}")
print(f"Parametri ottimizzati: {sum(p.numel() for p in model.parameters() if p.requires_grad) + sum(p.numel() for p in reconstruction_head.parameters() if p.requires_grad):,}")

✓ Modello pronto per il pretraining.
Learning rate: 0.001
Parametri ottimizzati: 201,795


In [ ]:
# E-10 — Pretraining: 1 epoca

model.train()
reconstruction_head.train()

num_epochs = 1

for epoch in range(num_epochs):

    total_loss = 0.0
    total_size = 0.0
    total_iat = 0.0
    total_direction = 0.0
    total_batches = 0

    for x_batch, pad_mask_batch, _ in train_loader:

        # Mascheriamo il 30% dei pacchetti validi
        mask_batch = create_pretraining_mask(
            pad_mask_batch,
            mask_ratio=0.30
        )

        # Copia dell'input e rimozione dei valori mascherati
        x_masked = x_batch.clone()
        x_masked[mask_batch] = 0.0

        # Azzera i gradienti
        optimizer_pretrain.zero_grad()

        # Encoder
        out = model(x_masked, pad_mask_batch)

        # Escludiamo il CLS
        packet_embeddings = out[:, 1:, :]

        # Reconstruction head
        pred_size, pred_iat, pred_direction = reconstruction_head(
            packet_embeddings
        )

        # Loss
        loss, loss_size, loss_iat, loss_direction = reconstruction_loss(
            pred_size,
            pred_iat,
            pred_direction,
            x_batch,
            mask_batch
        )

        # Backpropagation
        loss.backward()

        # Aggiornamento
        optimizer_pretrain.step()

        # Accumulo statistiche
        total_loss += loss.item()
        total_size += loss_size.item()
        total_iat += loss_iat.item()
        total_direction += loss_direction.item()
        total_batches += 1

        # Mostriamo il progresso ogni 200 batch
        if total_batches % 200 == 0:
            print(
                f"Batch {total_batches}/{len(train_loader)} "
                f"- Loss: {loss.item():.4f}"
            )

    # Media dell'epoca
    avg_loss = total_loss / total_batches
    avg_size = total_size / total_batches
    avg_iat = total_iat / total_batches
    avg_direction = total_direction / total_batches

    print("\n" + "=" * 50)
    print(f"EPOCA {epoch + 1} COMPLETATA")
    print(f"Loss totale:       {avg_loss:.6f}")
    print(f"Loss size:         {avg_size:.6f}")
    print(f"Loss IAT:          {avg_iat:.6f}")
    print(f"Loss direction:    {avg_direction:.6f}")
    print(f"Baseline:          {baseline_loss.item():.6f}")
    print("=" * 50)

Batch 200/1539 - Loss: 0.3030
Batch 400/1539 - Loss: 0.3303
Batch 600/1539 - Loss: 0.3378
Batch 800/1539 - Loss: 0.2987
Batch 1000/1539 - Loss: 0.3651
Batch 1200/1539 - Loss: 0.2643
Batch 1400/1539 - Loss: 0.2931

EPOCA 1 COMPLETATA
Loss totale:       0.345175
Loss size:         0.104862
Loss IAT:          0.150527
Loss direction:    0.089785
Baseline:          1.628992


In [ ]:
# E-10 — Salvataggio dell'encoder pre-addestrato

import os

pretrained_path = "/content/drive/MyDrive/Neural Collapse/pretrained_encoder.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "reconstruction_head_state_dict": reconstruction_head.state_dict(),
        "baseline_loss": baseline_loss.item(),
        "pretraining_loss": 0.535312
    },
    pretrained_path
)

print("✓ Encoder pre-addestrato salvato.")
print("Percorso:", pretrained_path)
print("Loss pretraining:", 0.535312)
print("Baseline:", baseline_loss.item())

✓ Encoder pre-addestrato salvato.
Percorso: /content/drive/MyDrive/Neural Collapse/pretrained_encoder.pt
Loss pretraining: 0.535312
Baseline: 1.6415491104125977


In [ ]:
# E-10 — Preparazione delle etichette label_20

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# Fit SOLO sul training set
y_train_20 = label_encoder.fit_transform(
    train_df["label_20"]
)

# Validation e test usando la stessa codifica
y_val_20 = label_encoder.transform(
    val_df["label_20"]
)

y_test_20 = label_encoder.transform(
    test_df["label_20"]
)

print("Numero classi:", len(label_encoder.classes_))
print("Classi:", label_encoder.classes_)

print("\nShape:")
print("y_train_20:", y_train_20.shape)
print("y_val_20:", y_val_20.shape)
print("y_test_20:", y_test_20.shape)

print("\n✓ Etichette label_20 preparate.")

Numero classi: 20
Classi: ['BitTorrent' 'Cridex' 'FTP' 'Facetime' 'Geodo' 'Gmail' 'Htbot' 'Miuref'
 'MySQL' 'Neris' 'Nsis-ay' 'Outlook' 'SMB' 'Shifu' 'Skype' 'Tinba' 'Virut'
 'Weibo' 'WorldOfWarcraft' 'Zeus']

Shape:
y_train_20: (393783,)
y_val_20: (84383,)
y_test_20: (56255,)

✓ Etichette label_20 preparate.


In [ ]:
# E-10 — Estrazione z_cls dal modello pre-addestrato

model.eval()
head.eval()

z_cls_pretrained, idx_pretrained = extract(
    model,
    head,
    extract_train_loader,
    "z_cls"
)

print("z_cls_pretrained:", z_cls_pretrained.shape)
print("idx_pretrained:", idx_pretrained.shape)

print("\n✓ z_cls del modello pre-addestrato estratto.")

z_cls_pretrained: (393783, 64)
idx_pretrained: (393783,)

✓ z_cls del modello pre-addestrato estratto.


In [ ]:
# E-10 — Encoder casuale di confronto

random_model = MiniNetEncoder(
    n_pkt=20,
    d=64,
    layers=4,
    heads=4
)

random_model.eval()

print("✓ Encoder casuale creato.")
print(
    "Parametri:",
    sum(p.numel() for p in random_model.parameters())
)

✓ Encoder casuale creato.
Parametri: 201600


/tmp/ipykernel_4471/1627283260.py:85: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.body = nn.TransformerEncoder(


In [ ]:
# E-10 — Estrazione z_cls dall'encoder casuale

random_z_cls, random_idx = extract(
    random_model,
    head,
    extract_train_loader,
    "z_cls"
)

print("random_z_cls:", random_z_cls.shape)
print("random_idx:", random_idx.shape)

print("\n✓ z_cls dell'encoder casuale estratto.")

random_z_cls: (393783, 64)
random_idx: (393783,)

✓ z_cls dell'encoder casuale estratto.


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_20 = label_encoder.fit_transform(
    df[df["split"] == "train"]["label_20"]
)

y_val_20 = label_encoder.transform(
    df[df["split"] == "val"]["label_20"]
)

y_test_20 = label_encoder.transform(
    df[df["split"] == "test"]["label_20"]
)

print("y_train_20:", y_train_20.shape)
print("Numero classi:", len(label_encoder.classes_))

y_train_20: (393783,)
Numero classi: 20


In [ ]:
from torch.utils.data import DataLoader

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False
)

print("✓ val_loader ricreato")
print("Batch:", len(val_loader))

✓ val_loader ricreato
Batch: 330


In [ ]:
from sklearn.linear_model import LogisticRegression

probe_random = LogisticRegression(
    max_iter=300,
    solver="saga",
    random_state=42
)

print("Addestramento probe RANDOM in corso...")

probe_random.fit(
    random_z_cls,
    y_train_20
)

print("✓ Probe RANDOM addestrato")

Addestramento probe RANDOM in corso...
✓ Probe RANDOM addestrato


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [ ]:
from sklearn.linear_model import LogisticRegression

probe_pretrained = LogisticRegression(
    max_iter=300,
    solver="saga",
    random_state=42
)

print("Addestramento probe PRETRAINED in corso...")

probe_pretrained.fit(
    z_cls_pretrained,
    y_train_20
)

print("✓ Probe PRETRAINED addestrato")

Addestramento probe PRETRAINED in corso...
✓ Probe PRETRAINED addestrato


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Estrazione z_cls della validation dal modello pretrained
z_cls_pretrained_val, idx_pretrained_val = extract(
    model,
    head,
    val_loader,
    "z_cls"
)

# Predizioni
pred_pretrained_val = probe_pretrained.predict(z_cls_pretrained_val)

# Metriche
acc_pretrained = accuracy_score(y_val_20, pred_pretrained_val)
f1_pretrained = f1_score(
    y_val_20,
    pred_pretrained_val,
    average="macro"
)

print("PRETRAINED — VALIDATION")
print(f"Accuracy: {acc_pretrained:.6f}")
print(f"F1 macro: {f1_pretrained:.6f}")

PRETRAINED — VALIDATION
Accuracy: 0.752687
F1 macro: 0.766570


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Estrazione z_cls della validation dal modello casuale
random_z_cls_val, random_idx_val = extract(
    random_model,
    head,
    val_loader,
    "z_cls"
)

# Predizioni
pred_random_val = probe_random.predict(random_z_cls_val)

# Metriche
acc_random = accuracy_score(y_val_20, pred_random_val)
f1_random = f1_score(
    y_val_20,
    pred_random_val,
    average="macro"
)

print("RANDOM — VALIDATION")
print(f"Accuracy: {acc_random:.6f}")
print(f"F1 macro: {f1_random:.6f}")

RANDOM — VALIDATION
Accuracy: 0.706837
F1 macro: 0.705539


In [ ]:
import os
import json
import numpy as np

SAVE_DIR = "/content/drive/MyDrive/Neural Collapse/E-10_results"
os.makedirs(SAVE_DIR, exist_ok=True)

# Risultati E-10
e10_results = {
    "baseline_loss": float(baseline_loss),
    "pretraining_loss": 0.535312,
    "pretrained_accuracy": float(acc_pretrained),
    "pretrained_f1_macro": float(f1_pretrained),
    "random_accuracy": float(acc_random),
    "random_f1_macro": float(f1_random),
    "criterion_1": bool(0.535312 < baseline_loss),
    "criterion_2": bool(f1_pretrained > f1_random)
}

with open(os.path.join(SAVE_DIR, "E10_results.json"), "w") as f:
    json.dump(e10_results, f, indent=4)

# Salva le rappresentazioni già calcolate
np.savez(
    os.path.join(SAVE_DIR, "E10_representations.npz"),
    z_cls_pretrained=z_cls_pretrained,
    random_z_cls=random_z_cls,
    z_cls_pretrained_val=z_cls_pretrained_val,
    random_z_cls_val=random_z_cls_val,
    y_train_20=y_train_20,
    y_val_20=y_val_20
)

print("✓ RISULTATI E-10 SALVATI SU GOOGLE DRIVE")
print()
print("Cartella:", SAVE_DIR)
print("File salvati:")
print("  - E10_results.json")
print("  - E10_representations.npz")
print()
print("Criterio 1:", "SUPERATO" if e10_results["criterion_1"] else "NON SUPERATO")
print("Criterio 2:", "SUPERATO" if e10_results["criterion_2"] else "NON SUPERATO")

✓ RISULTATI E-10 SALVATI SU GOOGLE DRIVE

Cartella: /content/drive/MyDrive/Neural Collapse/E-10_results
File salvati:
  - E10_results.json
  - E10_representations.npz

Criterio 1: SUPERATO
Criterio 2: SUPERATO


RIPRISTINO COMPLETO PER CONTINUARE DOPO LA PERDITA DEL RUNTIME:

In [1]:
# ============================================================
# RIPRISTINO COMPLETO — Neural Collapse
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

# ------------------------------------------------------------
# 1. GOOGLE DRIVE
# ------------------------------------------------------------

from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

print("✓ Google Drive disponibile")


# ------------------------------------------------------------
# 2. PERCORSI
# ------------------------------------------------------------

DATA_PATH = (
    "/content/drive/MyDrive/Neural Collapse/"
    "USTC-TFC2016/processed/"
    "ustc_tfc2016_preprocessed.parquet"
)

PRETRAINED_PATH = (
    "/content/drive/MyDrive/Neural Collapse/"
    "pretrained_encoder.pt"
)

E10_DIR = (
    "/content/drive/MyDrive/Neural Collapse/"
    "E-10_results"
)


# ------------------------------------------------------------
# 3. DATASET
# ------------------------------------------------------------

df = pd.read_parquet(DATA_PATH)

print("✓ Dataset caricato")
print("  Flussi:", len(df))
print("  Colonne:", len(df.columns))


# ------------------------------------------------------------
# 4. ENCODER
# ------------------------------------------------------------

class MiniNetEncoder(nn.Module):
    def __init__(self, n_pkt=20, d=64, layers=4, heads=4):
        super().__init__()

        self.inp = nn.Linear(3, d)

        self.cls = nn.Parameter(
            torch.zeros(1, 1, d)
        )

        self.pos = nn.Parameter(
            torch.zeros(1, n_pkt + 1, d)
        )

        enc = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=heads,
            dim_feedforward=4 * d,
            batch_first=True,
            norm_first=True
        )

        self.body = nn.TransformerEncoder(
            enc,
            num_layers=layers
        )

    def forward(self, x, pad_mask):

        h = self.inp(x)

        cls = self.cls.expand(
            h.size(0), -1, -1
        )

        h = torch.cat(
            [cls, h],
            dim=1
        )

        h = h + self.pos

        cls_mask = torch.zeros(
            (pad_mask.size(0), 1),
            dtype=torch.bool,
            device=pad_mask.device
        )

        pad_mask = torch.cat(
            [cls_mask, pad_mask],
            dim=1
        )

        return self.body(
            h,
            src_key_padding_mask=pad_mask
        )


# ------------------------------------------------------------
# 5. HEAD
# ------------------------------------------------------------

class Head(nn.Module):
    def __init__(self, d=64, K=20):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, d)
        )

        self.classifier = nn.Linear(d, K)

    def forward(self, z):
        h = self.mlp(z)
        return self.classifier(h), h


# ------------------------------------------------------------
# 6. SEQUENCE DATASET
# ------------------------------------------------------------

class FlowSequenceDataset(Dataset):

    def __init__(self, frame):

        self.direction = np.stack(
            frame["splt_direction"].to_numpy()
        ).astype(np.float32)

        self.ps = np.stack(
            frame["splt_ps"].to_numpy()
        ).astype(np.float32)

        self.piat = np.stack(
            frame["splt_piat_ms"].to_numpy()
        ).astype(np.float32)

        self.mask = np.stack(
            frame["splt_mask"].to_numpy()
        ).astype(np.bool_)

        self.indices = frame.index.to_numpy()

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):

        x = np.stack(
            [
                self.direction[i],
                self.ps[i],
                self.piat[i]
            ],
            axis=1
        )

        pad_mask = ~self.mask[i]

        return (
            torch.from_numpy(x),
            torch.from_numpy(pad_mask),
            int(self.indices[i])
        )


# ------------------------------------------------------------
# 7. SPLIT
# ------------------------------------------------------------

train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
test_df  = df[df["split"] == "test"]
probe_df = df[df["split"] == "probe"]

train_dataset = FlowSequenceDataset(train_df)
val_dataset   = FlowSequenceDataset(val_df)
test_dataset  = FlowSequenceDataset(test_df)
probe_dataset = FlowSequenceDataset(probe_df)

print("✓ Split ricreati")
print("  Train:", len(train_dataset))
print("  Val:  ", len(val_dataset))
print("  Test: ", len(test_dataset))
print("  Probe:", len(probe_dataset))


# ------------------------------------------------------------
# 8. DATALOADER
# ------------------------------------------------------------

batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

probe_loader = DataLoader(
    probe_dataset,
    batch_size=batch_size,
    shuffle=False
)

extract_train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=False
)

print("✓ DataLoader ricreati")


# ------------------------------------------------------------
# 9. FUNZIONE EXTRACT
# ------------------------------------------------------------

def extract(model, head, loader, which):

    model.eval()
    head.eval()

    representations = []
    indices = []

    with torch.inference_mode():

        for x, pad_mask, batch_indices in loader:

            out = model(x, pad_mask)

            if which == "z_cls":

                z = out[:, 0, :]

            elif which == "z_mean":

                valid = ~pad_mask

                z = (
                    (
                        out[:, 1:, :]
                        * valid.unsqueeze(-1)
                    ).sum(dim=1)
                    /
                    valid.sum(
                        dim=1,
                        keepdim=True
                    )
                )

            elif which == "h_pen":

                z_cls = out[:, 0, :]

                _, z = head(z_cls)

            else:

                raise ValueError(
                    "which deve essere "
                    "'z_cls', 'z_mean' oppure 'h_pen'"
                )

            representations.append(
                z.cpu().numpy()
            )

            indices.append(
                batch_indices.numpy()
            )

    representations = np.concatenate(
        representations,
        axis=0
    )

    indices = np.concatenate(
        indices,
        axis=0
    )

    return representations, indices


# ------------------------------------------------------------
# 10. MODELLO PRETRAINED
# ------------------------------------------------------------

model = MiniNetEncoder(
    n_pkt=20,
    d=64,
    layers=4,
    heads=4
)

checkpoint = torch.load(
    PRETRAINED_PATH,
    map_location="cpu"
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("✓ Encoder PRETRAINED ripristinato")


# ------------------------------------------------------------
# 11. RECONSTRUCTION HEAD
# ------------------------------------------------------------

class ReconstructionHead(nn.Module):

    def __init__(self, d=64):
        super().__init__()

        self.size_head = nn.Linear(d, 1)
        self.iat_head = nn.Linear(d, 1)
        self.direction_head = nn.Linear(d, 1)

    def forward(self, h):

        size = self.size_head(h).squeeze(-1)
        iat = self.iat_head(h).squeeze(-1)
        direction = self.direction_head(h).squeeze(-1)

        return size, iat, direction


reconstruction_head = ReconstructionHead(d=64)

if "reconstruction_head_state_dict" in checkpoint:

    reconstruction_head.load_state_dict(
        checkpoint["reconstruction_head_state_dict"]
    )

reconstruction_head.eval()

print("✓ Reconstruction head ripristinata")


# ------------------------------------------------------------
# 12. HEAD PER LE RAPPRESENTAZIONI
# ------------------------------------------------------------

head = Head(
    d=64,
    K=20
)

head.eval()

print("✓ Head ripristinata")


# ------------------------------------------------------------
# 13. ENCODER RANDOM
# ------------------------------------------------------------

torch.manual_seed(42)
np.random.seed(42)

random_model = MiniNetEncoder(
    n_pkt=20,
    d=64,
    layers=4,
    heads=4
)

random_model.eval()

print("✓ Encoder RANDOM ricreato")


# ------------------------------------------------------------
# 14. LABEL ENCODER
# ------------------------------------------------------------

label_encoder = LabelEncoder()

y_train_20 = label_encoder.fit_transform(
    train_df["label_20"]
)

y_val_20 = label_encoder.transform(
    val_df["label_20"]
)

y_test_20 = label_encoder.transform(
    test_df["label_20"]
)

print(
    "✓ Label encoder:",
    len(label_encoder.classes_),
    "classi"
)


# ------------------------------------------------------------
# 15. RISULTATI E-10
# ------------------------------------------------------------

E10_RESULTS_PATH = os.path.join(
    E10_DIR,
    "E10_results.json"
)

if os.path.exists(E10_RESULTS_PATH):

    with open(E10_RESULTS_PATH, "r") as f:
        e10_results = json.load(f)

    print("✓ Risultati E-10 caricati")

else:

    e10_results = None

    print("⚠ Risultati E-10 non trovati")


# ------------------------------------------------------------
# 16. RAPPRESENTAZIONI E-10
# ------------------------------------------------------------

E10_REP_PATH = os.path.join(
    E10_DIR,
    "E10_representations.npz"
)

if os.path.exists(E10_REP_PATH):

    e10_rep = np.load(
        E10_REP_PATH
    )

    z_cls_pretrained = e10_rep[
        "z_cls_pretrained"
    ]

    random_z_cls = e10_rep[
        "random_z_cls"
    ]

    z_cls_pretrained_val = e10_rep[
        "z_cls_pretrained_val"
    ]

    random_z_cls_val = e10_rep[
        "random_z_cls_val"
    ]

    print("✓ Rappresentazioni E-10 caricate")

else:

    print("⚠ Rappresentazioni E-10 non trovate")


# ------------------------------------------------------------
# 17. RIEPILOGO
# ------------------------------------------------------------

print()
print("=" * 55)
print("✓ RIPRISTINO COMPLETO TERMINATO")
print("=" * 55)

print("Dataset:", len(df), "flows")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))
print("Probe:", len(probe_df))

print(
    "Pretrained checkpoint:",
    "OK" if os.path.exists(PRETRAINED_PATH)
    else "MANCANTE"
)

print(
    "E-10 risultati:",
    "OK" if e10_results is not None
    else "MANCANTI"
)

print(
    "E-10 rappresentazioni:",
    "OK" if os.path.exists(E10_REP_PATH)
    else "MANCANTI"
)

print("=" * 55)

Mounted at /content/drive
✓ Google Drive disponibile
✓ Dataset caricato
  Flussi: 562549
  Colonne: 66
✓ Split ricreati
  Train: 393783
  Val:   84383
  Test:  56255
  Probe: 28128
✓ DataLoader ricreati


/tmp/ipykernel_2042/1627283260.py:85: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.body = nn.TransformerEncoder(


✓ Encoder PRETRAINED ripristinato
✓ Reconstruction head ripristinata
✓ Head ripristinata
✓ Encoder RANDOM ricreato
✓ Label encoder: 20 classi
✓ Risultati E-10 caricati
✓ Rappresentazioni E-10 caricate

✓ RIPRISTINO COMPLETO TERMINATO
Dataset: 562549 flows
Train: 393783
Val: 84383
Test: 56255
Probe: 28128
Pretrained checkpoint: OK
E-10 risultati: OK
E-10 rappresentazioni: OK
